In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import time
import csv
import json
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor
from threading import Lock # Cần cho việc ghi tăng dần nếu bạn áp dụng

# --- THIẾT LẬP CƠ BẢN ---
listings_scraped = 0 # KHÔNG DÙNG MAX_LISTINGS
MAX_WORKERS = 5 
all_data = []      
final_data = []    

# GIỮ NGUYÊN HEADER ĐỂ TRUY CẬP WEBSITE
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept-Language': 'vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7',
    'Referer': ''
}

# 🚨 DANH SÁCH BASE URLS MỚI CỦA BẠN 🚨
BASE_URLS = [
    "https://muaban.net/bat-dong-san/ban-nha-quan-ba-dinh-ha-noi?pt=43",
    "https://muaban.net/bat-dong-san/ban-nha-quan-cau-giay-ha-noi?pt=43",
    "https://muaban.net/bat-dong-san/ban-nha-quan-dong-da-ha-noi?pt=43",
    "https://muaban.net/bat-dong-san/ban-nha-quan-hai-ba-trung-ha-noi?pt=43",
    "https://muaban.net/bat-dong-san/ban-nha-quan-hoan-kiem-ha-noi?pt=43",
    "https://muaban.net/bat-dong-san/ban-nha-quan-thanh-xuan-ha-noi?pt=43",
]


# --- CÁC HÀM TIỆN ÍCH (GIỮ NGUYÊN LOGIC) ---

def is_short_time_text(s):
    """Kiểm tra xem chuỗi có phải là định dạng thời gian ngắn 'X giờ/phút/ngày trước' không."""
    if not s: return False
    s = s.strip()
    return bool(re.match(r'^\s*\d+\s*(giờ|phút|ngày)\s+trước\s*$', s, re.I))

def normalize_date(date_str):
    """Chuẩn hóa chuỗi ngày/giờ đăng thành định dạng YYYY-MM-DD."""
    if not date_str or date_str == 'N/A':
        return 'N/A'
    date_str = date_str.strip()
    now = datetime.now()

    # 1. Xử lý định dạng ISO 8601
    try:
        if re.match(r'\d{4}-\d{2}-\d{2}T', date_str):
            dt_obj = datetime.fromisoformat(date_str.replace('Z', '+00:00'))
            return dt_obj.date().isoformat()
    except ValueError:
        pass
    
    if is_short_time_text(date_str):
        match = re.search(r'(\d+)\s*(giờ|phút|ngày)\s+trước', date_str, re.I)
        if match:
            value = int(match.group(1))
            unit = match.group(2).lower()
            if 'phút' in unit: target_time = now - timedelta(minutes=value)
            elif 'giờ' in unit: target_time = now - timedelta(hours=value)
            elif 'ngày' in unit: target_time = now - timedelta(days=value)
            else: return 'N/A'
            return target_time.date().isoformat()
            
    # 3. Xử lý định dạng ngày cụ thể 'dd/mm/yyyy'
    date_match = re.match(r'(\d{1,2})[/-](\d{1,2})[/-](\d{4})', date_str)
    if date_match:
        day = int(date_match.group(1))
        month = int(date_match.group(2))
        year = int(date_match.group(3))
        try:
            return f"{day:02d}-{month:02d}-{year:04d}"
        except ValueError:
            return 'N/A'
    
    return 'N/A'

def make_absolute(href):
    if not href: return "N/A"
    href = href.strip()
    if href.startswith('http'): return href
    return "https://muaban.net" + href

def extract_detail_from_list(soup, label_keyword):
    """Tìm giá trị theo label (logic đã được giữ nguyên và bạn xác nhận là đúng)."""
    label_spans = soup.find_all('span', class_=re.compile(r'label', re.I))
    if not label_spans:
        return "N/A"

    for lbl in label_spans:
        full_label = "".join(lbl.stripped_strings)

        if re.search(label_keyword, full_label, re.I):
            value_tag = lbl.find_next_sibling(True)
        
            if value_tag:
                value = ' '.join(value_tag.stripped_strings)
                return value if value else 'N/A'

    return "N/A"

def get_description_text(soup_detail):
    """Tìm kiếm và trả về nội dung text đã làm sạch của phần mô tả."""
    header_block = soup_detail.find('div', class_=re.compile(r'bzqDYr', re.I))
    
    if header_block:
        parent_container = header_block.find_parent('div', class_=re.compile(r'eheBnp', re.I))
        if parent_container:
            description_block = parent_container.find('div', class_=re.compile(r'khOhZD', re.I))
            
            if description_block:
                return ' '.join(description_block.stripped_strings)
    
    return ""

def extract_from_description(description_text, keywords, search_name):
    """
    Trích xuất con số (cả số thập phân) từ text mô tả cho Số tầng, Phòng ngủ, và Phòng tắm.
    Logic đếm tầng phức tạp đã được loại bỏ.
    """
    if not description_text:
        return "N/A"
        
    match = None

    if search_name == 'so_tang':
        # Regex bắt số nguyên/thập phân (Ví dụ: 4,5 tầng)
        match = re.search(r'(\d+[\,\.]{0,1}\d*)\s*(tầng|lầu)', description_text, re.I)
        
        if match:
            # Lấy kết quả và CHUẨN HÓA thành dấu chấm để giữ giá trị thập phân
            return match.group(1).replace(',', '.').strip()
        
        return "N/A"

    elif search_name == 'phong_ngu':
        # Bắt số nguyên cho số phòng
        match = re.search(r'(\d{1,2})\s*(PN|phòng ngủ)', description_text, re.I)
    elif search_name == 'phong_tam':
        # Bắt số nguyên cho số phòng
        match = re.search(r'(\d{1,2})\s*(WC|phòng vệ sinh)', description_text, re.I)
    
    if match:
        return match.group(1).strip()
        
    return "N/A"

def scrape_detail_page(url, headers):
    """Truy cập trang chi tiết và cào tất cả các thuộc tính (bổ sung tìm trong Mô tả)."""
    detail_data = {
        'gia': 'N/A',
        'dia_chi': 'N/A',
        'dien_tich_dat': 'N/A',
        'phong_ngu': 'N/A',
        'phong_tam': 'N/A',
        'so_tang': 'N/A', 
        'phap_ly': 'N/A',
        'ngay_dang': 'N/A'
    }

    try:
        resp = requests.get(url, headers=headers, timeout=10)
        if resp.status_code != 200:
            print(f"Lỗi truy cập trang chi tiết {url}: Mã {resp.status_code}")
            return detail_data
        
        soup_detail = BeautifulSoup(resp.content, 'html.parser')

        # 1. Trích xuất Mô tả
        description_text = get_description_text(soup_detail)
        
        # --- PHẦN 1: HEADER (Giá, Địa chỉ) ---
        price_tag = soup_detail.find('div', class_='price')
        if price_tag: detail_data['gia'] = price_tag.get_text(strip=True)
             
        address_tag = soup_detail.find('div', class_=re.compile(r'address', re.I))
        if address_tag:
            span_icon = address_tag.find('span')
            if span_icon: span_icon.decompose()
            detail_data['dia_chi'] = address_tag.get_text(strip=True)
            

        # --- PHẦN 2: THÔNG TIN CƠ BẢN (VỚI LOGIC FALLBACK) ---
        
        # 1. Diện tích đất
        detail_data['dien_tich_dat'] = extract_detail_from_list(soup_detail, r'Diện tích sử dụng|Diện tích đất')
        
        # 2. Số tầng (Ưu tiên Thông tin cơ bản, nếu N/A, tìm trong Mô tả)
        detail_data['so_tang'] = extract_detail_from_list(soup_detail, r'Tổng số tầng|Số tầng') 
        if detail_data['so_tang'] == 'N/A':
             detail_data['so_tang'] = extract_from_description(description_text, r'(\d+[\,\.]{0,1}\d*)\s*(tầng|lầu)', 'so_tang')

        # 3. Phòng ngủ
        detail_data['phong_ngu'] = extract_detail_from_list(soup_detail, r'Số phòng ngủ')
        if detail_data['phong_ngu'] == 'N/A':
             detail_data['phong_ngu'] = extract_from_description(description_text, r'(\d{1,2})\s*(PN|phòng ngủ)', 'phong_ngu')
        
        # 4. Phòng tắm
        detail_data['phong_tam'] = extract_detail_from_list(soup_detail, r'Số phòng vệ sinh')
        if detail_data['phong_tam'] == 'N/A':
             detail_data['phong_tam'] = extract_from_description(description_text, r'(\d{1,2})\s*(WC|phòng vệ sinh)', 'phong_tam')
        
        # 5. Pháp lý
        detail_data['phap_ly'] = extract_detail_from_list(soup_detail, r'Giấy tờ pháp lý')
        
        # --- PHẦN 3: NGÀY BẮT ĐẦU (Ngày đăng) ---
        raw_date = 'N/A'
        
        start_date_label = soup_detail.find('span', string=re.compile(r'Ngày bắt đầu', re.I))
        if start_date_label:
            date_value_tag = start_date_label.find_next_sibling()
            if date_value_tag:
                raw_date = date_value_tag.get_text(strip=True)
        
        if raw_date == 'N/A':
             date_tag = soup_detail.find('div', class_=re.compile(r'date', re.I))
             if date_tag:
                 # Loại bỏ "Cập nhật:" và làm sạch
                 raw_date = date_tag.get_text(strip=True).replace('Cập nhật:', '').strip()
        
        # Gán giá trị sau khi chuẩn hóa 
        detail_data['ngay_dang'] = normalize_date(raw_date)
                
    except Exception as e:
        print(f"Lỗi cào trang chi tiết {url}: {e}")

    return detail_data

def scrape_listings_from_base_url(base_url, headers):
    """Cào danh sách URLs từ một BASE URL duy nhất (KHÔNG CÓ GIỚI HẠN MAX_LISTINGS)."""
    local_data = []
    current_page = 1
    
    while True: # Vòng lặp vô hạn, chỉ dừng khi hết trang hoặc lỗi
        # Sửa lỗi: Nếu URL đã có tham số (?), dùng & để phân trang
        if '?' in base_url:
            page_url = f"{base_url}&page={current_page}"
        else:
            page_url = f"{base_url}?page={current_page}"

        print("="*50)
        print(f"[{base_url.split('/')[-1]}] Đã cào: {len(local_data)} tin. | Đang cào Trang: {current_page} ")

        try:
            response = requests.get(page_url, headers=headers, timeout=10)
            
            if response.status_code == 200:
                soup = BeautifulSoup(response.content, 'html.parser')

                all_links = soup.find_all('a', href=re.compile(r'^/bat-dong-san/', re.I)) 
                
                new_listings_count = 0
                for link_tag in all_links:
                    h3_tag = link_tag.find('h3')
                    if h3_tag: 
                        title = h3_tag.get_text(strip=True)
                        url_detail_path = link_tag.get('href')
                        detail_url = make_absolute(url_detail_path)
                        
                        title_cleaned = re.sub(r'\s*\d+,\d+\s*m2\s*$', '', title, flags=re.I)

                        local_data.append({
                            'Tieu_de': title_cleaned,
                            'url_detail': detail_url
                        })

                        new_listings_count += 1
                
                if new_listings_count == 0:
                    print(f"[{base_url.split('/')[-1]}] Không tìm thấy tin đăng mới. Đã hết trang/dữ liệu.")
                    break # Hết dữ liệu trên Base URL này

                current_page += 1
                time.sleep(1) 

            elif response.status_code == 404:
                print(f"[{base_url.split('/')[-1]}] Lỗi 404. Đã hết dữ liệu.")
                break
            else:
                print(f"[{base_url.split('/')[-1]}] Lỗi: Mã trạng thái {response.status_code}. Tạm dừng 10 giây.")
                time.sleep(10)

        except requests.exceptions.RequestException as e:
            print(f"[{base_url.split('/')[-1]}] Lỗi kết nối: {e}. Tạm dừng 10 giây.")
            time.sleep(10)
            
    return local_data

# === Giai đoạn 1: CÀO DANH SÁCH TỪ NHIỀU BASE URL ===
print("--- Bắt đầu Cào DANH SÁCH TỪ NHIỀU BASE URLS ---")
for base_url in BASE_URLS:
    
    new_data = scrape_listings_from_base_url(base_url, headers)
    
    all_data.extend(new_data)
    listings_scraped = len(all_data)
    print(f"\n--- Tổng tin đã cào: {listings_scraped} ---")


# === Giai đoạn 2: CÀO CHI TIẾT TỪ URL VÀ CHUẨN HÓA DỮ LIỆU (ĐA LUỒNG) ===
print("\n--- Bắt đầu cào chi tiết (ĐA LUỒNG) và Lưu Tăng Dần ---")

# --- HÀM GHI CSV TĂNG DẦN MỚI ---
def write_to_csv_incrementally(data_record, file_name, fields):
    """Ghi một bản ghi vào CSV. Ghi header chỉ khi file chưa tồn tại."""
    from threading import Lock
    file_lock = Lock()
    
    file_exists = False
    try:
        with open(file_name, 'r', encoding='utf-8') as f:
            if f.read(1):  
                file_exists = True
    except FileNotFoundError:
        pass 

    try:
        with file_lock: # Sử dụng Lock để đảm bảo an toàn khi ghi đa luồng
            with open(file_name, 'a', newline='', encoding='utf-8') as output_file:
                dict_writer = csv.DictWriter(output_file, fieldnames=fields)
                
                if not file_exists:
                    dict_writer.writeheader()
                
                dict_writer.writerow(data_record)
            
            return True
    except Exception as e:
        print(f"❌ Lỗi khi ghi file CSV: {e}")
        return False
# ------------------------------------

# --- THIẾT LẬP CSV ---
CSV_FILE_NAME = 'muaban.net.csv'
# 🚨 URL CHI TIẾT ĐÃ BỊ LOẠI BỎ KHỎI DANH SÁCH CÁC TRƯỜNG CẦN LƯU 🚨
CSV_FIELDS = ['Tieu_de','gia','dia_chi','dien_tich_dat','phong_ngu','phong_tam','so_tang','phap_ly','ngay_dang'] 


def process_detail_item_and_save(item):
    """Hàm wrapper cho cào chi tiết để chạy trong luồng và lưu ngay lập tức."""
    detail_url = item['url_detail']
    
    if detail_url == "N/A":
        return None

    # Tạm dừng ngắn để tránh quá tải server/bị chặn khi cào chi tiết
    time.sleep(1) 
    
    detail_info = scrape_detail_page(detail_url, headers)

    ordered_record = {
        'Tieu_de': item['Tieu_de'],
        'gia': detail_info.get('gia', 'N/A'),
        'dia_chi': detail_info.get('dia_chi', 'N/A'),
        'dien_tich_dat': detail_info.get('dien_tich_dat', 'N/A'),
        'phong_ngu': detail_info.get('phong_ngu', 'N/A'),
        'phong_tam': detail_info.get('phong_tam', 'N/A'),
        'so_tang': detail_info.get('so_tang', 'N/A'), 
        'phap_ly': detail_info.get('phap_ly', 'N/A'),
        'ngay_dang': detail_info['ngay_dang'],
    }
    
    if write_to_csv_incrementally(ordered_record, CSV_FILE_NAME, CSV_FIELDS):
        print(f"✅ Ghi thành công tin: {ordered_record['Tieu_de']}...")
        return ordered_record
    return None # Nếu ghi file thất bại

results_processed = 0
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # Sử dụng executor.map thay vì submit/future để xử lý dễ dàng hơn
    results = executor.map(process_detail_item_and_save, all_data)
    
    for result in results:
        if result:
            results_processed += 1

print("="*50)
print(f"Quá trình cào và lưu tăng dần hoàn tất. Đã xử lý và lưu {results_processed} tin.")
